# TQP 본실험 v1.0 (HW2)

HW1 계획서(5절)대로 한 번 실행하는 본실험입니다. 위에서부터 순서대로 실행합니다. 생성 결과는 셀 단위로 Drive에 저장되므로 런타임이 끊겨도 다시 실행하면 이어서 진행합니다.

| 절 | 내용 | 자원 |
|---|---|---|
| 1 | 설정, 평가 데이터 고정(해시) | CPU |
| 2 | 생성: 방법 2 × 조건 3 × (평가 템플릿, 자유 문장, 개발 세트) | **T4** |
| 3 | 채점(탐침 v0.5), H1 · H2 · H3 · H3b, 보조 지표 | CPU |
| 4 | 실제 로그 검증: AV2 train 150개 로그, 검색 집합 변화 | CPU |
| 5 | 수작업 자료: 검출 정밀도(사건 80개), B2 프로그램 40개 | 본인 |

In [1]:
# 1-1. 설정
import os, sys, json, time, random, subprocess
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

REPO = Path('/content/drive/MyDrive/NLP/tqp_repo')            # 코드 (Drive)
OUT = Path('/content/drive/MyDrive/NLP/AV2_TQP_TEG_Results/main_v1') # 결과 (Drive)
CACHE_ROOT = Path('/content/drive/MyDrive/NLP/AV2_TEG_Pilot_Data')  # AV2 로그 캐시 (파일럿과 공유)
FREEFORM_CSV = REPO / 'data' / 'freeform.csv'                  # 본인 검토를 마친 자유 문장
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
BATCH = 16
SEED = 2026
N_TRAIN_LOGS = 150
OUT.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels', 'boto3', 'openpyxl'], check=True)
sys.path.insert(0, str(REPO))
import importlib, numpy as np, pandas as pd
import tqp_teg as T, tqp_nl as N, tqp_eval as E, tqp_run as R, tqp_stats as S
for m in (T, N, E, R, S): importlib.reload(m)
print('engine', T.VERSION, '| detector', type(T.DEFAULT_DET).__name__, '| out', OUT)

Mounted at /content/drive
engine 0.4.0 | detector DetectorConfigV04 | out /content/drive/MyDrive/NLP/AV2_TQP_TEG_Results/main_v1


In [2]:
# 1-2. 평가 데이터 고정: 개발 세트(288), 평가 템플릿(864), 자유 문장 + 누출 검사 + 해시
dev = E.build_dev_set()
tpl = E.build_eval_templates()
free = E.load_freeform(FREEFORM_CSV) if FREEFORM_CSV.exists() else None
for df, p in ((dev, 'd'), (tpl, 't')) + (((free, 'f'),) if free is not None else ()):
    df['sid'] = [f'{p}{i:04d}' for i in range(len(df))]
evalset = pd.concat([tpl] + ([free] if free is not None else []), ignore_index=True)
leak = E.leakage_report(evalset)
assert leak.empty, leak
assert not set(evalset.sentence) & set(dev.sentence)
manifest = {'engine': T.VERSION, 'model': MODEL_ID, 'seed': SEED,
            'dev_sha256': E.frame_hash(dev), 'eval_tpl_sha256': E.frame_hash(tpl),
            'eval_free_sha256': E.frame_hash(free) if free is not None else None,
            'prompts_sha256': E.prompts_hash(),
            'n': {'dev': len(dev), 'eval_tpl': len(tpl), 'eval_free': 0 if free is None else len(free)},
            'clusters': {'dev': dev.cluster.nunique(), 'eval_tpl': tpl.cluster.nunique(),
                         'eval_free': 0 if free is None else free.cluster.nunique()}}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=1))
for name, df in (('dev', dev), ('eval_tpl', tpl)) + ((('eval_free', free),) if free is not None else ()):
    df.to_csv(OUT / f'data_{name}.csv', index=False)
print(json.dumps(manifest, indent=1))
print(evalset.groupby(['split', 'gold_relation', 'order_type']).size())

{
 "engine": "0.4.0",
 "model": "Qwen/Qwen2.5-1.5B-Instruct",
 "seed": 2026,
 "dev_sha256": "7102eabaa6c3090e4b7dfba01f87bdf6ff6b0de4d192a1e7a32ccbdc51579740",
 "eval_tpl_sha256": "1e4209b602e31a58cf0024ace9ac91969785706733715182ac427ec22c7ea275",
 "eval_free_sha256": "64ff02138a8ec7c322156300a7c7c99c464152449f23ee735e3c4b5f65d6dacc",
 "prompts_sha256": "987707e71dd130314bcfbd8fb1795f1795d86b89f0eae84fbfec0d208c6963c7",
 "n": {
  "dev": 288,
  "eval_tpl": 864,
  "eval_free": 50
 },
 "clusters": {
  "dev": 72,
  "eval_tpl": 96,
  "eval_free": 11
 }
}
split      gold_relation  order_type  
eval_free  THEN           congruent        19
                          incongruent      16
           WHILE          simultaneous     15
eval_tpl   THEN           congruent       432
                          incongruent     324
           WHILE          simultaneous    108
dtype: int64


## 2. 생성 (T4 필요). 셀마다 Drive에 저장되며, 같은 문장·프롬프트면 다시 생성하지 않습니다.

In [3]:
# 2-1. 모델 로드 + 생성: 평가 템플릿 → 자유 문장 → 개발 세트 (예상: 템플릿 40~60분, 개발 15~20분)
import torch
assert torch.cuda.is_available(), '런타임 유형을 T4 GPU로 바꾸십시오.'
model, tok, device = N.load_model(MODEL_ID)
RAW = OUT / 'raw'
raws = {}
splits = [('eval_tpl', tpl)] + ([('eval_free', free)] if free is not None else []) + [('dev', dev)]
t0 = time.time()
for split, data in splits:
    for cond in N.MAIN_CONDITIONS:
        for method in N.METHODS:
            raws[(split, method, cond)] = R.generate_cell(model, tok, device, RAW, split, method, cond, data, batch_size=BATCH)
print(f'generation done in {(time.time() - t0) / 60:.1f} min')

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  generated eval_tpl/P/zero_shot: 864 in 130s -> raw_eval_tpl_P_zero_shot_82315e0a78cfcad5.csv
  generated eval_tpl/B2/zero_shot: 864 in 969s -> raw_eval_tpl_B2_zero_shot_6eef2fa886c3b3a4.csv
  generated eval_tpl/P/balanced: 864 in 183s -> raw_eval_tpl_P_balanced_0ffd44f251727180.csv
  generated eval_tpl/B2/balanced: 864 in 553s -> raw_eval_tpl_B2_balanced_9463a37409861165.csv
  generated eval_tpl/P/counterbalanced: 864 in 214s -> raw_eval_tpl_P_counterbalanced_fbe9b74b559bdac7.csv
  generated eval_tpl/B2/counterbalanced: 864 in 689s -> raw_eval_tpl_B2_counterbalanced_3295795db02b94e2.csv
  generated eval_free/P/zero_shot: 50 in 9s -> raw_eval_free_P_zero_shot_01105460dd01a1f9.csv
  generated eval_free/B2/zero_shot: 50 in 69s -> raw_eval_free_B2_zero_shot_6ee2cefbe3aa02f1.csv
  generated eval_free/P/balanced: 50 in 12s -> raw_eval_free_P_balanced_ee995c9e8c8d4057.csv
  generated eval_free/B2/balanced: 50 in 40s -> raw_eval_free_B2_balanced_ddd2ab3bcda1bb5e.csv
  generated eval_free/P/c

## 3. 채점과 사전 지정 분석

In [4]:
# 3-1. 채점 (탐침 v0.5: 해석 탐침 5 + 구현 탐침 8)
RAW = OUT / 'raw'
def load_raw(split, method, cond, data):
    key = R.cell_key(split, method, cond, data.sentence.tolist())
    return pd.read_csv(RAW / f'raw_{split}_{method}_{cond}_{key}.csv', keep_default_na=False)
scored = []
for split, data in [('eval_tpl', tpl)] + ([('eval_free', free)] if free is not None else []) + [('dev', dev)]:
    for cond in N.MAIN_CONDITIONS:
        for method in N.METHODS:
            scored.append(R.score_cell(data, load_raw(split, method, cond, data), method, cond))
scored = pd.concat(scored, ignore_index=True)
scored.to_csv(OUT / 'scored_all.csv', index=False)
ev = scored[scored.split.isin(['eval_tpl', 'eval_free'])].copy()
dv = scored[scored.split == 'dev'].copy()
# P는 실행기가 판정하므로 해석 정확도 = 판정 정확도여야 함
p = ev[ev.method == 'P']
assert (p.interp_correct == p.judge_correct).all()
print('rows', len(scored), '| eval', len(ev), '| dev', len(dv))

rows 7212 | eval 5484 | dev 1728


In [5]:
# 3-2. H1: 조건별 P − B2 (관계 균형 해석 정확도), 군집 부트스트랩 + 군집 sign-flip 순열검정, Holm 보정
h1 = []
for cond in N.MAIN_CONDITIONS:
    d = S.paired_effect(ev, (ev.method == 'P') & (ev.condition == cond), (ev.method == 'B2') & (ev.condition == cond))
    h1.append({'condition': cond, 'Acc_bal_P': d['acc_a'], 'Acc_bal_B2': d['acc_b'], 'D': d['D'],
               'CI_low': d['ci_low'], 'CI_high': d['ci_high'], 'p_perm': d['p_perm'],
               'n_sent': d['n_sentences'], 'n_clusters': d['n_clusters']})
h1 = pd.DataFrame(h1)
h1['p_holm'] = S.holm(h1.p_perm)
h1['H1'] = np.where((h1.p_holm < 0.05) & (h1.D > 0), 'holds', np.where(h1.p_holm < 0.05, 'reversed', 'inconclusive'))
print('##### H1 (평가 세트 = 템플릿 + 자유 문장)'); print(h1.round(4).to_string(index=False))
# 분할별 (보고용)
rows = []
for split in ['eval_tpl', 'eval_free']:
    for cond in N.MAIN_CONDITIONS:
        for m in N.METHODS:
            sub = ev[(ev.split == split)]
            if len(sub):
                c = S.accuracy_ci(sub, (sub.method == m) & (sub.condition == cond))
                rows.append({'split': split, 'condition': cond, 'method': m, **c})
acc_tab = pd.DataFrame(rows)
print('\n##### 관계 균형 해석 정확도 [95% CI], 분할별'); print(acc_tab.round(3).to_string(index=False))

##### H1 (평가 세트 = 템플릿 + 자유 문장)
      condition  Acc_bal_P  Acc_bal_B2       D  CI_low  CI_high  p_perm  n_sent  n_clusters  p_holm           H1
      zero_shot     0.0025      0.0135 -0.0110 -0.0224  -0.0019  0.0361     914         107  0.0780 inconclusive
       balanced     0.5992      0.4555  0.1436  0.0488   0.2381  0.0260     914         107  0.0780 inconclusive
counterbalanced     0.4314      0.3327  0.0987  0.0425   0.1503  0.1979     914         107  0.1979 inconclusive

##### 관계 균형 해석 정확도 [95% CI], 분할별
    split       condition method   acc  ci_low  ci_high   n
 eval_tpl       zero_shot      P 0.003   0.000    0.007 864
 eval_tpl       zero_shot     B2 0.015   0.006    0.026 864
 eval_tpl        balanced      P 0.596   0.552    0.639 864
 eval_tpl        balanced     B2 0.462   0.365    0.561 864
 eval_tpl counterbalanced      P 0.437   0.374    0.495 864
 eval_tpl counterbalanced     B2 0.337   0.304    0.372 864
eval_free       zero_shot      P 0.000   0.000    0.000  50
eva

In [6]:
# 3-3. H2: 로지스틱 GEE (method × condition + relation), 군집 = 문장 군집, exchangeable, 가중치 1/(2 n_relation)
h2 = S.gee_interaction(ev)
print('##### H2 interaction Wald test'); print({k: h2[k] for k in ('wald_chi2', 'df', 'p_value', 'n_obs', 'n_clusters', 'dep_param')})
print(pd.DataFrame({'coef': h2['params'], 'se': h2['bse']}).to_string())

##### H2 interaction Wald test
{'wald_chi2': 9.10066179689533, 'df': 2, 'p_value': 0.010563708290783656, 'n_obs': 5484, 'n_clusters': 107, 'dep_param': 0.10001056661488071}
                                                                                           coef      se
Intercept                                                                               -3.8977  0.3406
C(method, Treatment('B2'))[T.P]                                                         -1.4569  0.6095
C(condition, Treatment('zero_shot'))[T.balanced]                                         3.9991  0.3796
C(condition, Treatment('zero_shot'))[T.counterbalanced]                                  3.4758  0.3443
C(gold_relation)[T.WHILE]                                                               -0.4692  0.2255
C(method, Treatment('B2'))[T.P]:C(condition, Treatment('zero_shot'))[T.balanced]         2.0475  0.6824
C(method, Treatment('B2'))[T.P]:C(condition, Treatment('zero_shot'))[T.counterbalanced]  1.8809  0.

In [7]:
# 3-4. 보조 지표: 비가중 정확도, 유효성, 판정 정확도, B2 구현 오류, 표현별 사건·관계·방향 정확도
def sec(g):
    out = {'n': len(g), 'acc_unweighted': g.interp_correct.mean(), 'acc_bal': S.balanced_accuracy(g),
           'valid': g.valid.mean(), 'judge_acc_bal': S.balanced_accuracy(g, 'judge_correct'),
           'impl_error': g.impl_error.mean(),
           'acc_THEN': g[g.gold_relation == 'THEN'].interp_correct.mean(),
           'acc_WHILE': g[g.gold_relation == 'WHILE'].interp_correct.mean()}
    return pd.Series(out)
secondary = ev.groupby(['condition', 'method']).apply(sec)
print('##### 보조 지표 (평가 세트)'); print(secondary.round(3).to_string())
print('\n##### 오류 유형 분포 (탐침 기준, 공통 정의)')
print(pd.crosstab([ev.condition, ev.method], ev.error_type).to_string())
print('\n##### 어순 일치/불일치 × 접속사 (THEN), 해석 정확도')
then = ev[ev.gold_relation == 'THEN']
print(then.pivot_table(index=['condition', 'method'], columns='order_type', values='interp_correct', aggfunc='mean').round(3).to_string())
print(then[then.split == 'eval_tpl'].pivot_table(index=['condition', 'method'], columns='template', values='interp_correct', aggfunc='mean').round(2).to_string())

##### 보조 지표 (평가 세트)
                            n  acc_unweighted  acc_bal  valid  judge_acc_bal  impl_error  acc_THEN  acc_WHILE
condition       method                                                                                       
balanced        B2      914.0           0.545    0.456  0.999          0.456       0.000     0.578      0.333
                P       914.0           0.556    0.599  0.834          0.599       0.000     0.540      0.659
counterbalanced B2      914.0           0.558    0.333  1.000          0.333       0.000     0.641      0.024
                P       914.0           0.354    0.431  0.833          0.431       0.000     0.326      0.537
zero_shot       B2      914.0           0.018    0.014  0.375          0.002       0.014     0.019      0.008
                P       914.0           0.004    0.003  0.004          0.003       0.000     0.005      0.000

##### 오류 유형 분포 (탐침 기준, 공통 정의)
error_type              correct  direction  direction+types  event_ty

/tmp/ipykernel_782/2080370788.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  secondary = ev.groupby(['condition', 'method']).apply(sec)


In [8]:
# 3-5. H3 (P 방향 오류의 하위 유형), H3b (B2: 관계 유형 오류 vs 방향 오류)
h3_rows = []
pe = ev[(ev.method == 'P') & (ev.gold_relation == 'THEN')]
for cond in N.MAIN_CONDITIONS:
    sub = pe[pe.condition == cond]
    den = sub.h3_subtype.notna()
    for cong in (False, True):
        d2 = den & (sub.congruent == cong)
        for st in sorted(sub[d2].h3_subtype.unique()):
            c = S.proportion_ci(sub, d2, sub.h3_subtype == st)
            h3_rows.append({'condition': cond, 'congruent': cong, 'subtype': st, **c})
h3 = pd.DataFrame(h3_rows)
print('##### H3: P에서 사건 유형은 맞고 THEN 방향이 틀린 출력의 하위 유형 (비율 [95% CI])')
print(h3.round(3).to_string(index=False))
h3b = []
for cond in N.MAIN_CONDITIONS:
    sub = ev[(ev.method == 'B2') & (ev.condition == cond)]
    den = sub.error_type.isin(['relation_type', 'direction', 'direction+types'])
    c_rel = S.proportion_ci(sub, den, sub.error_type == 'relation_type')
    h3b.append({'condition': cond, 'relation_type': int((sub.error_type == 'relation_type').sum()),
                'direction': int(sub.error_type.isin(['direction', 'direction+types']).sum()),
                'share_relation_type': c_rel['share'], 'CI_low': c_rel['ci_low'], 'CI_high': c_rel['ci_high'],
                'other_errors': int((~sub.error_type.isin(['correct', 'relation_type', 'direction', 'direction+types'])).sum())})
h3b = pd.DataFrame(h3b)
print('\n##### H3b: B2 관계 유형 오류 vs 방향 오류 (관계 유형 오류 비율 [95% CI])'); print(h3b.round(3).to_string(index=False))

##### H3: P에서 사건 유형은 맞고 THEN 방향이 틀린 출력의 하위 유형 (비율 [95% CI])
      condition  congruent                  subtype  share  ci_low  ci_high   k   n
       balanced      False               args_error  0.463   0.209    0.753  38  82
       balanced      False          double_reversal  0.537   0.247    0.791  44  82
       balanced       True args_reversed(congruent)  1.000   1.000    1.000 101 101
counterbalanced      False               args_error  0.347   0.118    0.616  26  75
counterbalanced      False          double_reversal  0.653   0.384    0.882  49  75
counterbalanced       True args_reversed(congruent)  0.842   0.664    0.990 101 120
counterbalanced       True slot_reversal(congruent)  0.158   0.010    0.336  19 120

##### H3b: B2 관계 유형 오류 vs 방향 오류 (관계 유형 오류 비율 [95% CI])
      condition  relation_type  direction  share_relation_type  CI_low  CI_high  other_errors
      zero_shot             36         29                0.554   0.418    0.691           833
       balanced         

In [9]:
# 3-6. 개발 세트 연속성 확인 (파일럿 v0.4 결과와 비교, 같은 288문장)
dev_tab = dv.groupby(['condition', 'method']).apply(lambda g: pd.Series({'acc_bal': S.balanced_accuracy(g), 'acc': g.interp_correct.mean()}))
print(dev_tab.round(3).to_string())
pilot = sorted(Path('/content/drive/MyDrive/NLP/AV2_TQP_TEG_Results').glob('v04_val_*/llm_predictions_v04.csv'))
if pilot:
    pv = pd.read_csv(pilot[-1])
    pv['method'] = pv.method.map({'P (TQP)': 'P', 'B2 (Python)': 'B2'})
    m = dv.merge(pv[['sentence', 'variant', 'method', 'raw_output']].rename(columns={'variant': 'condition', 'raw_output': 'pilot_output'}),
                 on=['sentence', 'condition', 'method'], how='left')
    print('파일럿 출력과 동일한 비율:', m.groupby(['condition', 'method']).apply(lambda g: (g.raw_output == g.pilot_output).mean()).round(3).to_dict())

/tmp/ipykernel_782/1418210733.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dev_tab = dv.groupby(['condition', 'method']).apply(lambda g: pd.Series({'acc_bal': S.balanced_accuracy(g), 'acc': g.interp_correct.mean()}))


                        acc_bal    acc
condition       method                
balanced        B2        0.779  0.854
                P         0.931  0.913
counterbalanced B2        0.435  0.726
                P         0.808  0.681
zero_shot       B2        0.069  0.073
                P         0.215  0.177
파일럿 출력과 동일한 비율: {('balanced', 'B2'): 1.0, ('balanced', 'P'): 1.0, ('counterbalanced', 'B2'): 1.0, ('counterbalanced', 'P'): 1.0, ('zero_shot', 'B2'): 1.0, ('zero_shot', 'P'): 1.0}


/tmp/ipykernel_782/1418210733.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print('파일럿 출력과 동일한 비율:', m.groupby(['condition', 'method']).apply(lambda g: (g.raw_output == g.pilot_output).mean()).round(3).to_dict())


In [10]:
# 3-7. 저장
report = {'manifest': manifest, 'H1': h1.to_dict('records'), 'acc_by_split': acc_tab.to_dict('records'),
          'H2': h2, 'secondary': secondary.reset_index().to_dict('records'),
          'error_types': pd.crosstab([ev.condition, ev.method], ev.error_type).reset_index().to_dict('records'),
          'H3': h3.to_dict('records'), 'H3b': h3b.to_dict('records'), 'dev': dev_tab.reset_index().to_dict('records')}
(OUT / 'report_main.json').write_text(json.dumps(report, indent=1, default=str))
print('saved', OUT / 'report_main.json')

saved /content/drive/MyDrive/NLP/AV2_TQP_TEG_Results/main_v1/report_main.json


## 4. 실제 로그 검증 (AV2 train에서 무작위 150개 로그, CPU)

In [11]:
# 4-1. 로그 확보 + 검출(v0.4) + 그래프 + 두 실행기 일치 확인
import pickle
ids_path = OUT / 'train_log_ids.json'
if ids_path.exists():
    train_ids = json.loads(ids_path.read_text())
else:
    all_ids = T.list_s3_logs('train')
    train_ids = sorted(random.Random(SEED).sample(all_ids, N_TRAIN_LOGS))
    ids_path.write_text(json.dumps(train_ids))
folders, acq = T.ensure_av2_logs(CACHE_ROOT, 'train', N_TRAIN_LOGS, workers=16, log_ids=train_ids)
print({k: v for k, v in acq.items() if k != 'failed'}, 'failed', len(acq['failed']))
events, universe, signals, infos = [], set(), {}, []
t0 = time.time()
for f in folders:
    evs, info, uni, sig = T.load_av2_log(f)
    events += evs; universe |= uni; signals.update(sig); infos.append(info)
print(f'detection {time.time() - t0:.1f}s | tracks {len(universe)} | events {len(events)}', pd.Series([e.type for e in events]).value_counts().to_dict())
t0 = time.time(); graph = T.TemporalEventGraph(events, universe); print(f'graph {time.time() - t0:.1f}s', graph.summary()['relation_edges_by_type'])
gexec = T.GraphQueryExecutor(graph)
ver, _ = T.verify_and_benchmark(events, universe, T.all_queries(), repeat=10)
print({k: v for k, v in ver.items() if k != 'per_query'})
with open(OUT / 'train_events.pkl', 'wb') as fh:
    pickle.dump({'events': events, 'universe': universe, 'signals': signals, 'infos': infos}, fh)

{'split': 'train', 'requested': 150, 'ready': 150, 'from_cache': 0, 'downloaded': 150, 'note': None} failed 0
detection 14.8s | tracks 3110 | events 3751 {'ACCELERATE': 1422, 'DECELERATE': 1091, 'STOP': 965, 'TURN': 273}
graph 0.7s {'THEN': 1320, 'WHILE': 192}
{'version': '0.4.0', 'relation_config': {'then_max_gap_s': 5.0, 'then_overlap_tolerance_s': 0.5, 'while_min_overlap_s': 0.5, 'while_min_ratio': 0.8, 'clip_grid_step_s': 0.1, 'clip_grid_max_points': 300000}, 'graph': {'track_nodes': 3110, 'event_nodes': 3751, 'has_event_edges': 3751, 'relation_edges': 1512, 'relation_edges_by_type': {'THEN': 1320, 'WHILE': 192}, 'relation_edges_by_pattern': {'ACCELERATE THEN ACCELERATE': 90, 'ACCELERATE THEN DECELERATE': 188, 'ACCELERATE THEN STOP': 6, 'ACCELERATE THEN TURN': 23, 'DECELERATE THEN ACCELERATE': 141, 'DECELERATE THEN DECELERATE': 59, 'DECELERATE THEN STOP': 224, 'DECELERATE THEN TURN': 16, 'STOP THEN ACCELERATE': 387, 'STOP THEN DECELERATE': 11, 'STOP THEN STOP': 83, 'STOP THEN TURN'

In [12]:
# 4-2. 해석 오류가 검색 집합 (log_id, track_id)를 얼마나 바꾸는가 (정답 결과가 빈 질의 제외)
track_dicts = {}
for e in events:
    track_dicts.setdefault(e.track_key, []).append({'type': e.type, 'start': e.start, 'end': e.end})
empty_tracks = [k for k in universe if k not in track_dicts]
t0 = time.time()
rt = R.retrieval_table(ev, gexec, track_dicts, empty_tracks)
print(f'retrieval {time.time() - t0:.0f}s')
rt = rt.merge(ev[['sid', 'method', 'condition', 'gold_relation', 'interp_correct', 'split']], on=['sid', 'method', 'condition'])
rt.to_csv(OUT / 'retrieval.csv', index=False)
ne = rt[rt.gold_n > 0]
print('정답 결과가 있는 문장:', ne.sid.nunique(), '/', rt.sid.nunique())
print(ne.groupby(['condition', 'method'])[['same_set', 'jaccard']].mean().round(3).to_string())
print(ne.groupby(['method', 'interp_correct'])[['same_set', 'jaccard']].mean().round(3).to_string())
# 참고: B2 정답 코드(해석·구현 모두 정답)를 원시 구간에 실행했을 때 그래프 정답과의 일치 (표현 차이: 잘린 경계 처리)
ref = []
for q in T.all_queries():
    gs = frozenset((m.log_id, m.track_id) for m in gexec.retrieve(q))
    bs = R.b2_retrieve(N.b2_reference_code(q), track_dicts, empty_tracks)
    ref.append({'query': str(q), 'graph_n': len(gs), 'ref_b2_n': len(bs), 'jaccard': R.jaccard(gs, bs)})
print(pd.DataFrame(ref).to_string(index=False))

retrieval 193s
정답 결과가 있는 문장: 914 / 914
                        same_set  jaccard
condition       method                   
balanced        B2      0.500548    0.546
                P       0.666667    0.563
counterbalanced B2      0.554705    0.568
                P       0.425756    0.370
zero_shot       B2      0.003096    0.032
                P            1.0    0.004
                       same_set  jaccard
method interp_correct                   
B2     False                0.0    0.021
       True            0.944227    0.987
P      False                0.0    0.011
       True                 1.0    1.000
                      query  graph_n  ref_b2_n  jaccard
             TURN THEN TURN        3         3 1.000000
       TURN THEN DECELERATE       26        26 1.000000
       TURN THEN ACCELERATE        8         8 1.000000
             TURN THEN STOP        7         7 1.000000
            TURN WHILE TURN        0         0 1.000000
      TURN WHILE DECELERATE       22       

## 5. 수작업 자료 만들기 (판정은 본인이 직접)

In [13]:
# 5-1. 검출 정밀도: 유형별 사건 20개 (총 80개) 무작위 표본 → PDF(사건마다 1쪽) + 판정 시트
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
rng = random.Random(SEED)
by_type = {t: [e for e in events if e.type == t] for t in T.EVENT_TYPES}
sample = [e for t in T.EVENT_TYPES for e in rng.sample(by_type[t], min(20, len(by_type[t])))]
by_track = {}
for e in events:
    by_track.setdefault(e.track_key, []).append(e)
with PdfPages(OUT / 'detector_check.pdf') as pdf:
    for i, e in enumerate(sample, 1):
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        T.plot_track(e.track_key, [e], signals[e.track_key], axes, title_extra=f'#{i} {e.type} [{e.start:.1f}-{e.end:.1f}s]')
        for ax in axes[1:]:
            ax.axvline(e.start, ls='--', c='k'); ax.axvline(e.end, ls='--', c='k')
        fig.suptitle(f'#{i}  {e.type}  {e.start:.1f}–{e.end:.1f}s', fontsize=12)
        pdf.savefig(fig); plt.close(fig)
pd.DataFrame([{'no': i, 'type': e.type, 'log_id': e.log_id, 'track_id': e.track_id, 'start': round(e.start, 2),
               'end': round(e.end, 2), 'clipped': e.clipped, 'verdict': ''} for i, e in enumerate(sample, 1)]
             ).to_excel(OUT / 'detector_check.xlsx', index=False)
print('saved', OUT / 'detector_check.pdf', OUT / 'detector_check.xlsx', len(sample), 'events')

saved /content/drive/MyDrive/NLP/AV2_TQP_TEG_Results/main_v1/detector_check.pdf /content/drive/MyDrive/NLP/AV2_TQP_TEG_Results/main_v1/detector_check.xlsx 80 events


In [14]:
# 5-2. B2 프로그램 수작업 점검: 탐침상 정답 20개 + 오답 20개 (평가 세트, 세 조건 통합 무작위)
b2 = ev[(ev.method == 'B2') & ev.valid]
pick = pd.concat([b2[b2.interp_correct].sample(20, random_state=SEED), b2[~b2.interp_correct].sample(20, random_state=SEED)]).sample(frac=1, random_state=SEED)
sheet = pd.DataFrame({'item': [f'b{i:02d}' for i in range(len(pick))], 'sid': pick.sid.values, 'condition': pick.condition.values,
                      'sentence': pick.sentence.values,
                      'gold': (pick.gold_first + ' ' + pick.gold_relation + ' ' + pick.gold_second).values,
                      'code': [N.extract_code(t) for t in pick.raw_output], 'probe_says': np.where(pick.interp_correct, 'correct', 'incorrect'),
                      'my_judgment': '', 'note': ''})
sheet.to_excel(OUT / 'b2_inspection.xlsx', index=False)
print('saved', OUT / 'b2_inspection.xlsx')

saved /content/drive/MyDrive/NLP/AV2_TQP_TEG_Results/main_v1/b2_inspection.xlsx
